# Spatial2D latent distance probe

This notebook is a quick diagnostic for two failure modes in Spatial2D ABC inference:

1. The `[initial, final]` temporal latent is dominated by the shared initial frame.
2. The latent `pairwise_cosine` metric is not sensitive enough to simulator differences.

It does not run ABC. It loads one trained checkpoint, builds the same Spatial2D system used by inference, samples a small number of parameter vectors, and compares latent distances against raw-grid differences.

## Problem

Recent ABC runs show distances flattening near a narrow epsilon range while accepted parameters remain poorly constrained. This probe tests whether the plateau is mainly caused by the shared initial frame dominating the temporal latent representation, or by the latent distance metric being insensitive to meaningful final-state simulation differences.

In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
import rootutils

PROJECT_ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from src.viaABC.systems import Spatial2D
from src.viaABC.metrics import pairwise_cosine, l2_distance

print(f"project root: {PROJECT_ROOT}")
print(f"torch: {torch.__version__}, cuda: {torch.cuda.is_available()}")

## Configuration

Set `RUN_DIR` to the training run whose checkpoint and observation config you want to inspect. Keep `N_THETA` modest; every theta simulates all four 1200x1200 samples and runs one encoder pass.

In [ ]:
RUN_DIR = Path("/insomnia001/depts/iicd/users/kz2537/viaABC/run/train/spatial2D/2026-06-03_11-34-14_bs10_acc2_nw2")
CHECKPOINT_SUBSTR = "last"
POOLING_METHOD = "no_cls"
METRIC = "pairwise_cosine"

N_THETA = 24
SEED = 12345
PRIOR_LOW = np.array([0.0, 0.0], dtype=np.float64)
PRIOR_HIGH = np.array([1.0, 1.0], dtype=np.float64)

OUTPUT_DIR = RUN_DIR / "latent_distance_probe"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_DIR, OUTPUT_DIR, DEVICE

## Load checkpoint and Spatial2D system

In [ ]:
def load_training_config(run_dir: Path):
    cfg_path = run_dir / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    return OmegaConf.load(cfg_path)


def load_model_and_transform(run_dir: Path, checkpoint_substr: str, device: torch.device):
    train_cfg = load_training_config(run_dir)
    model = instantiate(train_cfg.model)

    ckpt_dir = run_dir / "checkpoints"
    matches = sorted(ckpt_dir.glob(f"*{checkpoint_substr}*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"No checkpoint matching {checkpoint_substr!r} in {ckpt_dir}")
    ckpt_path = matches[-1]
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    fixed_state = {}
    for key, value in checkpoint["state_dict"].items():
        if key.startswith("model."):
            key = key[len("model."):]
        if key.startswith("_orig_mod."):
            key = key[len("_orig_mod."):]
        fixed_state[key] = value

    missing, unexpected = model.model.load_state_dict(fixed_state, strict=False)
    if missing:
        print("missing keys:", missing)
    if unexpected:
        print("unexpected keys:", unexpected)

    model.to(device)
    model.eval()
    transform_cfg = train_cfg.data.get("transform", None)
    transform = instantiate(transform_cfg) if transform_cfg is not None else None
    return train_cfg, model, transform, ckpt_path


@contextmanager
def use_training_observation_samples(train_cfg):
    samples = OmegaConf.to_container(train_cfg.data.observation_samples, resolve=True)
    original = Spatial2D._load_spatial2d_samples
    Spatial2D._load_spatial2d_samples = staticmethod(lambda: samples)
    try:
        yield
    finally:
        Spatial2D._load_spatial2d_samples = original


train_cfg, model, transform, ckpt_path = load_model_and_transform(RUN_DIR, CHECKPOINT_SUBSTR, DEVICE)
system_cfg = OmegaConf.create(OmegaConf.to_container(train_cfg.system, resolve=True))
system_cfg.pooling_method = POOLING_METHOD
system_cfg.metric = METRIC
system_cfg.mu = PRIOR_LOW.tolist()
system_cfg.sigma = PRIOR_HIGH.tolist()

with use_training_observation_samples(train_cfg):
    system = instantiate(system_cfg, model=model, transform=transform)

print("checkpoint:", ckpt_path)
print("sample ids:", OmegaConf.to_container(train_cfg.data.observation_sample, resolve=True))
print("initial grids:", system._initial_grids.shape)
print("observation grids:", system._observation_grids.shape)
print("encoded observation:", system.encoded_observational_data.shape)
print("time_space:", system.time_space, "num_frames:", system.num_frames)

## Distance helpers

`pairwise_cosine` averages over all latent tokens. For this temporal model the token order is frame-major, so we can split tokens back into frame 0 and frame 1 and inspect each contribution separately.

In [ ]:
def cosine_distance_tokens(x: np.ndarray, y: np.ndarray, eps: float = 1e-8) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    x_norm = x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)
    y_norm = y / (np.linalg.norm(y, axis=-1, keepdims=True) + eps)
    return float(1.0 - np.sum(x_norm * y_norm, axis=-1).mean())


def split_frame_tokens(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z)
    inner_model = system.model.model
    frames = int(inner_model.patch_embed.t_grid_size)
    grid = int(inner_model.patch_embed.grid_size)
    spatial_tokens = grid * grid
    expected_tokens = frames * spatial_tokens
    tokens = z
    if tokens.shape[1] == expected_tokens + 1:
        tokens = tokens[:, 1:, :]
    if tokens.shape[1] != expected_tokens:
        raise ValueError(f"Cannot split {tokens.shape[1]} tokens into {frames} frames x {spatial_tokens} spatial tokens")
    return tokens.reshape(tokens.shape[0], frames, spatial_tokens, tokens.shape[-1])


def latent_distance_breakdown(obs_z: np.ndarray, sim_z: np.ndarray) -> dict[str, float]:
    obs_z = np.asarray(obs_z)
    sim_z = np.asarray(sim_z)
    obs_frames = split_frame_tokens(obs_z)
    sim_frames = split_frame_tokens(sim_z)
    return {
        "latent_pairwise_cosine": float(1.0 - pairwise_cosine(obs_z, sim_z)),
        "latent_token_cosine": cosine_distance_tokens(obs_z, sim_z),
        "latent_l2": float(l2_distance(obs_z, sim_z)),
        "latent_frame0_cosine": cosine_distance_tokens(obs_frames[:, 0], sim_frames[:, 0]),
        "latent_frame1_cosine": cosine_distance_tokens(obs_frames[:, 1], sim_frames[:, 1]),
    }


def raw_grid_metrics(initial: np.ndarray, observed: np.ndarray, simulated: np.ndarray) -> dict[str, float]:
    initial = np.asarray(initial)
    observed = np.asarray(observed)
    simulated = np.asarray(simulated)
    obs_changed = observed != initial
    sim_changed = simulated != initial
    intersection = np.logical_and(obs_changed, sim_changed).sum()
    union = np.logical_or(obs_changed, sim_changed).sum()
    obs_hist = np.bincount(observed.reshape(-1), minlength=6) / observed.size
    sim_hist = np.bincount(simulated.reshape(-1), minlength=6) / simulated.size
    return {
        "raw_final_mismatch": float(np.mean(observed != simulated)),
        "raw_obs_changed_frac": float(np.mean(obs_changed)),
        "raw_sim_changed_frac": float(np.mean(sim_changed)),
        "raw_changed_iou": float(intersection / union) if union else math.nan,
        "raw_class_hist_l1": float(np.abs(obs_hist - sim_hist).sum()),
    }


def encode_label_pairs(pairs: np.ndarray) -> np.ndarray:
    with torch.inference_mode():
        return system.get_latent(system.preprocess(pairs))


def compare_one_pair(sample_index: int, pair: np.ndarray, label: str, theta=None) -> dict[str, object]:
    obs_z = system.encoded_observational_data[sample_index:sample_index + 1]
    sim_z = encode_label_pairs(pair[np.newaxis, ...])
    row = {
        "label": label,
        "theta_alpha": float(theta[0]) if theta is not None else math.nan,
        "theta_beta": float(theta[1]) if theta is not None else math.nan,
        "sample_index": sample_index,
    }
    row.update(latent_distance_breakdown(obs_z, sim_z))
    row.update(raw_grid_metrics(system._initial_grids[sample_index], system._observation_grids[sample_index], pair[-1]))
    return row

## Positive and negative controls

These controls calibrate the scale. If all controls are near the ABC epsilon plateau, the latent/metric is likely saturated.

In [ ]:
control_rows = []
num_samples = system._initial_grids.shape[0]
rng = np.random.default_rng(SEED)

for i in range(num_samples):
    initial = system._initial_grids[i]
    observed = system._observation_grids[i]
    j = (i + 1) % num_samples

    control_pairs = {
        "identity_obs_vs_obs": np.stack([initial, observed], axis=0),
        "null_final_obs_vs_initial": np.stack([initial, initial], axis=0),
        "cross_initial_same_final": np.stack([system._initial_grids[j], observed], axis=0),
        "same_initial_cross_final": np.stack([initial, system._observation_grids[j]], axis=0),
        "same_initial_random_final": np.stack([initial, rng.integers(0, 6, size=initial.shape, dtype=np.uint8)], axis=0),
    }
    for label, pair in control_pairs.items():
        control_rows.append(compare_one_pair(i, pair, label))

controls_df = pd.DataFrame(control_rows)
controls_df.to_csv(OUTPUT_DIR / "controls.csv", index=False)
controls_df.groupby("label")[["latent_pairwise_cosine", "latent_frame0_cosine", "latent_frame1_cosine", "raw_final_mismatch", "raw_changed_iou"]].agg(["mean", "std", "min", "max"])

## Random-theta simulator probe

This measures whether latent distance tracks raw simulator differences across parameter proposals. If raw differences vary but latent distances barely move, the metric/representation is not discriminative enough for ABC.

In [ ]:
rng = np.random.default_rng(SEED)
random_thetas = rng.uniform(PRIOR_LOW, PRIOR_HIGH, size=(N_THETA, 2))
anchor_thetas = np.array([
    [0.0, 0.0],
    [0.1, 0.1],
    [0.25, 0.25],
    [0.5, 0.5],
    [0.75, 0.75],
    [1.0, 1.0],
], dtype=np.float64)
thetas = np.vstack([anchor_thetas, random_thetas])
thetas[:10], len(thetas)

In [ ]:
probe_rows = []

for theta_index, theta in enumerate(thetas):
    sims, status = system.simulate_for_inference(theta)
    if status != 0:
        print(f"theta {theta_index} failed: {theta}")
        continue
    with torch.inference_mode():
        sim_z_all = system.get_latent(system.preprocess(sims))
    for sample_index in range(num_samples):
        obs_z = system.encoded_observational_data[sample_index:sample_index + 1]
        sim_z = sim_z_all[sample_index:sample_index + 1]
        row = {
            "label": "theta_probe",
            "theta_index": theta_index,
            "theta_alpha": float(theta[0]),
            "theta_beta": float(theta[1]),
            "sample_index": sample_index,
        }
        row.update(latent_distance_breakdown(obs_z, sim_z))
        row.update(raw_grid_metrics(system._initial_grids[sample_index], system._observation_grids[sample_index], sims[sample_index, -1]))
        probe_rows.append(row)
    if (theta_index + 1) % 5 == 0:
        print(f"processed {theta_index + 1}/{len(thetas)} theta")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / "theta_probe.csv", index=False)
probe_df.head()

## Summaries

Look first at ranges, then at rank correlations. A useful latent metric should have a non-trivial range and should correlate with raw final/transition differences.

In [ ]:
summary_cols = [
    "latent_pairwise_cosine",
    "latent_frame0_cosine",
    "latent_frame1_cosine",
    "latent_l2",
    "raw_final_mismatch",
    "raw_changed_iou",
    "raw_class_hist_l1",
]
probe_df[summary_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

In [ ]:
corr = probe_df[summary_cols].corr(method="spearman")
corr.to_csv(OUTPUT_DIR / "spearman_correlation.csv")
corr

In [ ]:
per_theta = probe_df.groupby("theta_index", as_index=False).agg({
    "theta_alpha": "first",
    "theta_beta": "first",
    "latent_pairwise_cosine": "mean",
    "latent_frame0_cosine": "mean",
    "latent_frame1_cosine": "mean",
    "raw_final_mismatch": "mean",
    "raw_changed_iou": "mean",
    "raw_class_hist_l1": "mean",
})
per_theta.to_csv(OUTPUT_DIR / "theta_probe_aggregated.csv", index=False)
per_theta.sort_values("latent_pairwise_cosine").head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(probe_df["raw_final_mismatch"], probe_df["latent_pairwise_cosine"], s=18, alpha=0.7)
axes[0].set_xlabel("raw final mismatch")
axes[0].set_ylabel("latent pairwise cosine distance")

axes[1].scatter(probe_df["raw_final_mismatch"], probe_df["latent_frame1_cosine"], s=18, alpha=0.7)
axes[1].set_xlabel("raw final mismatch")
axes[1].set_ylabel("latent frame1 cosine distance")

axes[2].scatter(probe_df["latent_frame0_cosine"], probe_df["latent_frame1_cosine"], s=18, alpha=0.7)
axes[2].set_xlabel("latent frame0 distance")
axes[2].set_ylabel("latent frame1 distance")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "latent_probe_scatter.png", dpi=160)
plt.show()

## Reading the result

- If `null_final_obs_vs_initial` is close to the ABC epsilon plateau, the final observation frame contributes weakly.
- If `latent_frame0_cosine` is near zero but `latent_frame1_cosine` varies meaningfully, the aggregate metric is being diluted by the shared initial frame.
- If raw metrics vary but `latent_frame1_cosine` and `latent_pairwise_cosine` barely vary, the latent representation or cosine metric is not sensitive enough.
- If latent L2 correlates with raw differences but pairwise cosine does not, try `metric=l2` or a final-frame-only metric for inference.
- If cross-sample controls are also small, check whether the encoder has collapsed to coarse/global features.

In [ ]:
result_manifest = {
    "run_dir": str(RUN_DIR),
    "checkpoint": str(ckpt_path),
    "n_theta": int(len(thetas)),
    "pooling_method": POOLING_METHOD,
    "metric": METRIC,
    "outputs": {
        "controls": str(OUTPUT_DIR / "controls.csv"),
        "theta_probe": str(OUTPUT_DIR / "theta_probe.csv"),
        "theta_probe_aggregated": str(OUTPUT_DIR / "theta_probe_aggregated.csv"),
        "spearman_correlation": str(OUTPUT_DIR / "spearman_correlation.csv"),
        "scatter": str(OUTPUT_DIR / "latent_probe_scatter.png"),
    },
}
with (OUTPUT_DIR / "manifest.json").open("w") as f:
    json.dump(result_manifest, f, indent=2)
result_manifest